In [3]:
import pandas as pd
import re
from tqdm import tqdm

# 1. Cấu hình thanh tiến trình cho Pandas
tqdm.pandas(desc="Đang ủi phẳng text")

def clean_pdf_text(text):
    """
    Hàm xử lý text từ PDF: nối lại các câu bị đứt đoạn do rớt dòng,
    nhưng vẫn bảo toàn cấu trúc các đoạn văn (paragraph).
    """
    # Bỏ qua nếu dữ liệu bị rỗng (NaN/Null)
    if not isinstance(text, str): 
        return ""
    
    # Bước 1: Bảo vệ các đoạn văn thực sự (những chỗ có 2 dấu \n trở lên)
    # Tạm thời thay thế chúng bằng một thẻ đánh dấu đặc biệt
    text = re.sub(r'\n{2,}', '<PARAGRAPH_MARKER>', text)
    
    # Bước 2: Dọn rác
    # Những dấu \n đứng trơ trọi 1 mình chính là rớt dòng giữa câu -> Thay bằng dấu cách
    text = re.sub(r'\n', ' ', text)
    
    # Bước 3: Phục hồi đoạn văn
    # Trả lại dấu xuống dòng chuẩn (\n\n) vào đúng vị trí thẻ đánh dấu ban đầu
    text = text.replace('<PARAGRAPH_MARKER>', '\n\n')
    
    # Bước 4: Dọn dẹp các khoảng trắng thừa (nhiều dấu cách liền nhau)
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

def process_and_clean_csv(input_filepath, output_filepath, text_column):
    print(f"Đang đọc dữ liệu từ: {input_filepath}")
    
    try:
        # Đọc file CSV
        df = pd.read_csv(input_filepath)
    except Exception as e:
        print(f"Lỗi khi đọc file: {e}")
        return
    
    print(f"Tổng số dòng cần xử lý: {len(df)}")
    
    # Điền chuỗi rỗng vào các ô bị thiếu dữ liệu (tránh lỗi khi chạy Regex)
    df[text_column] = df[text_column].fillna("")
    
    # Áp dụng hàm làm sạch lên toàn bộ cột nội dung
    df[text_column] = df[text_column].progress_apply(clean_pdf_text)
    
    # Lưu lại kết quả ra file CSV mới
    df.to_csv(output_filepath, index=False, encoding="utf-8")
    print(f"✅ Đã xử lý xong! File sạch được lưu tại: {output_filepath}")

# ==========================================
# KHU VỰC THỰC THI CHƯƠNG TRÌNH
# ==========================================

if __name__ == "__main__":
    # ⚠️ QUAN TRỌNG: Bạn hãy thay đổi 3 biến dưới đây cho khớp với hệ thống của bạn
    
    FILE_DAU_VAO = "../data/clean/tthcm.csv"  # Đường dẫn file CSV bị rớt dòng
    FILE_DAU_RA = "tthcm.csv"   # Tên file mới bạn muốn lưu
    COT_NOI_DUNG = "content"                         # Tên cột chứa chữ trong file CSV (VD: "text" hoặc "content")
    
    process_and_clean_csv(FILE_DAU_VAO, FILE_DAU_RA, COT_NOI_DUNG)

Đang đọc dữ liệu từ: ../data/clean/tthcm.csv
Tổng số dòng cần xử lý: 131


Đang ủi phẳng text: 100%|██████████████████████████████████████████████████████████| 131/131 [00:00<00:00, 8088.05it/s]

✅ Đã xử lý xong! File sạch được lưu tại: tthcm.csv
